# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [ ]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
# Section 1 checks (GUIDE step 1): counts, session leakage, boxes per class, visualize 6 train images
import json, random
from collections import Counter
from pathlib import Path
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

DATA = Path('data')
def read_jsonl(p):
    return [json.loads(l) for l in open(p, encoding='utf-8') if l.strip()]

splits = {s: read_jsonl(DATA / 'splits' / f'{s}.jsonl') for s in ('train', 'val', 'test')}
meta = json.load(open(DATA / 'splits' / 'meta.json', encoding='utf-8'))
print('dataset_version:', meta['dataset_version'])

# 1) images / boxes per split
for s, rows in splits.items():
    boxes = Counter(o['label'] for r in rows for o in r['objects'])
    sizes = Counter((r['width'], r['height']) for r in rows)
    n_empty = sum(1 for r in rows if not r['objects'])
    n_sess = len({r['session'] for r in rows})
    print(f"{s:5s} images={len(rows):4d} sessions={n_sess:3d} empty_images={n_empty:3d} "
          f"boxes={dict(boxes)} total_boxes={sum(boxes.values())} sizes={dict(sizes)}")
    assert len(rows) == meta['counts'][s]['images']

# 2) no session (and no image) shared between splits
sess = {s: {r['session'] for r in rows} for s, rows in splits.items()}
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    inter = sess[a] & sess[b]
    print(f'sessions {a} & {b}: {len(inter)}')
    assert not inter, inter
ids = [r['image_id'] for rows in splits.values() for r in rows]
assert len(ids) == len(set(ids)), 'duplicate image_id across splits'
print('OK: no session / image_id shared between splits')

# 3) box size: median box area as % of frame (small boxes are hard)
areas = sorted(o['yolo'][2] * o['yolo'][3] for r in splits['train'] for o in r['objects'])
print(f"train median box area = {100 * areas[len(areas) // 2]:.2f}% of frame")

# 4) draw 6 train images with GT boxes (YOLO xc,yc,w,h normalized -> pixel corners)
COLORS = {'pothole': 'red', 'crack': 'yellow', 'manhole': 'cyan'}
random.seed(0)
sample = random.sample([r for r in splits['train'] if r['objects']], 6)
fig, axes = plt.subplots(2, 3, figsize=(18, 7))
for ax, r in zip(axes.flat, sample):
    img = Image.open(DATA / r['image_path']).convert('RGB')
    W, H = img.size
    d = ImageDraw.Draw(img)
    for o in r['objects']:
        xc, yc, w, h = o['yolo']
        box = [(xc - w / 2) * W, (yc - h / 2) * H, (xc + w / 2) * W, (yc + h / 2) * H]
        d.rectangle(box, outline=COLORS[o['label']], width=3)
        d.text((box[0] + 3, box[1] + 3), o['label'], fill=COLORS[o['label']])
    ax.imshow(img)
    ax.set_title(f"{r['image_id']} | {r['session']} | {len(r['objects'])} boxes", fontsize=9)
    ax.axis('off')
plt.tight_layout()
Path('report/figures').mkdir(parents=True, exist_ok=True)
plt.savefig('report/figures/step1_train_samples.png', dpi=80)
plt.show()

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
# Section 2 (GUIDE step 2): YOLO labels -> JSON answer on the 0-1000 scale + prompt
import json
from scripts.score import parse_output, iou_xyxy   # the referee's own parser / IoU (read-only use)

def yolo_to_box(label, yolo):
    xc, yc, w, h = yolo                                   # normalized [0,1], center + size
    x1, y1, x2, y2 = (min(1000, max(0, round(v * 1000)))
                      for v in (xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2))
    # keep tiny boxes valid: score.parse_output rejects the WHOLE answer if any x2<=x1 or y2<=y1
    if x2 <= x1: x1, x2 = (x1, x1 + 1) if x1 < 1000 else (999, 1000)
    if y2 <= y1: y1, y2 = (y1, y1 + 1) if y1 < 1000 else (999, 1000)
    return {'label': label, 'bbox_2d': [x1, y1, x2, y2]}

def objects_to_answer(objects):
    boxes = [yolo_to_box(o['label'], o['yolo']) for o in objects]
    boxes.sort(key=lambda b: (b['bbox_2d'][1], b['bbox_2d'][0]))   # top->bottom, left->right: stable order
    return json.dumps(boxes, separators=(',', ':'))                 # "[]" if no object

PROMPT = (
    "Detect all road damage in this dashcam image. Classes: pothole, crack, manhole.\n"
    "Return ONLY a JSON list, with no extra text and no markdown, in the form "
    '[{"label": "pothole|crack|manhole", "bbox_2d": [x1, y1, x2, y2]}].\n'
    "bbox_2d coordinates are integers on a 0-1000 scale relative to the image width and height "
    "(x1,y1 = top-left corner, x2,y2 = bottom-right corner). If there is no damage, return []."
)

# Check 1: every answer in every split parses with the referee's parser, keeps all boxes,
# and the integer 0-1000 box stays close to the exact GT box the referee uses
worst_iou, worst_px, n_boxes, lens = 1.0, 0.0, 0, []
for s, rows in splits.items():
    for r in rows:
        ans = objects_to_answer(r['objects'])
        lens.append(len(ans))
        parsed = parse_output(ans)
        assert parsed is not None and len(parsed) == len(r['objects']), (s, r['image_id'], ans)
        for o in r['objects']:
            xc, yc, w, h = o['yolo']
            exact = ((xc - w / 2) * 1000, (yc - h / 2) * 1000, (xc + w / 2) * 1000, (yc + h / 2) * 1000)
            got = yolo_to_box(o['label'], o['yolo'])['bbox_2d']
            worst_iou = min(worst_iou, iou_xyxy(exact, tuple(got)))
            scale = (r['width'], r['height'], r['width'], r['height'])
            worst_px = max(worst_px, max(abs(g - e) * k / 1000 for g, e, k in zip(got, exact, scale)))
            n_boxes += 1
print(f"{n_boxes} boxes in {len(lens)} answers all parse OK with score.parse_output")
print(f"worst IoU(int 0-1000 box, exact GT) = {worst_iou:.3f}; worst corner error = {worst_px:.2f} px")
print(f"answer length: max={max(lens)} chars, mean={sum(lens) / len(lens):.0f} chars")

# Check 2: round trip on the busiest train image: JSON 0-1000 -> pixel must sit on the original YOLO box
r = max(splits['train'], key=lambda r: len(r['objects']))
ans = objects_to_answer(r['objects'])
print('\nPROMPT:\n' + PROMPT)
print(f"\nanswer for {r['image_id']} ({len(r['objects'])} boxes):\n{ans}")
img = Image.open(DATA / r['image_path']).convert('RGB')
W, H = img.size
d = ImageDraw.Draw(img)
for o in r['objects']:                          # original YOLO -> pixel (thick green)
    xc, yc, w, h = o['yolo']
    d.rectangle([(xc - w / 2) * W, (yc - h / 2) * H, (xc + w / 2) * W, (yc + h / 2) * H], outline='lime', width=5)
for b in json.loads(ans):                       # JSON 0-1000 -> pixel (thin red), should lie on the green box
    x1, y1, x2, y2 = b['bbox_2d']
    d.rectangle([x1 * W / 1000, y1 * H / 1000, x2 * W / 1000, y2 * H / 1000], outline='red', width=1)
    d.text((x1 * W / 1000 + 3, y1 * H / 1000 + 3), b['label'], fill='red')
plt.figure(figsize=(12, 7))
plt.imshow(img)
plt.axis('off')
plt.title(f"Round trip {r['image_id']}: green = original YOLO box, red = JSON 0-1000 -> pixel")
plt.savefig('report/figures/step2_roundtrip.png', dpi=80, bbox_inches='tight')
plt.show()

In [ ]:
# Diagnose GT boxes whose integer 0-1000 version differs from the exact box the referee uses
# Found: 1 zero-height crack box (train, vlcsnap-00058) and 45 boxes extending outside the frame
bad = []
for s, rows in splits.items():
    for r in rows:
        for o in r['objects']:
            xc, yc, w, h = o['yolo']
            exact = ((xc - w / 2) * 1000, (yc - h / 2) * 1000, (xc + w / 2) * 1000, (yc + h / 2) * 1000)
            got = tuple(yolo_to_box(o['label'], o['yolo'])['bbox_2d'])
            v = iou_xyxy(exact, got)
            if v < 0.9:
                bad.append((round(v, 3), s, r['image_id'], o['label'], [round(x, 4) for x in o['yolo']],
                            [round(x, 1) for x in exact], list(got)))
print(len(bad), 'boxes with IoU(int, exact) < 0.9')
for b in sorted(bad)[:20]:
    print(b)
out_of_frame = sum(1 for rows in splits.values() for r in rows for o in r['objects']
                   if min(o['yolo'][0] - o['yolo'][2] / 2, o['yolo'][1] - o['yolo'][3] / 2) < -1e-6
                   or max(o['yolo'][0] + o['yolo'][2] / 2, o['yolo'][1] + o['yolo'][3] / 2) > 1 + 1e-6)
zero_size = sum(1 for rows in splits.values() for r in rows for o in r['objects'] if o['yolo'][2] <= 0 or o['yolo'][3] <= 0)
print('boxes extending outside the frame:', out_of_frame, '| zero-size boxes:', zero_size)

## 3. E0: zero-shot

In [ ]:
# Section 3 (GUIDE step 3): E0 zero-shot — load Qwen3-VL-4B in 4-bit (ViT + lm_head kept fp16)
import gc, time
import torch
from tqdm.auto import tqdm
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration

MODEL_ID = 'Qwen/Qwen3-VL-4B-Instruct'
Path('work').mkdir(exist_ok=True)

def load_model():
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                               bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
                               llm_int8_skip_modules=['visual', 'lm_head'])
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID, quantization_config=quant, dtype=torch.float16, device_map={'': 0})
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    processor.tokenizer.padding_side = 'left'                 # left pad for batched generation
    return model, processor

def build_messages(img, prompt=PROMPT):
    return [{'role': 'user', 'content': [{'type': 'image', 'image': img}, {'type': 'text', 'text': prompt}]}]

@torch.no_grad()
def predict(model, processor, rows, out_path, batch_size=4, max_new_tokens=256, prompt=PROMPT):
    """Greedy-decode every row, append {"image_id","raw"} per line. Resume-safe: skips ids already in out_path."""
    out_path = Path(out_path)
    done = {json.loads(l)['image_id'] for l in open(out_path, encoding='utf-8')} if out_path.exists() else set()
    todo = [r for r in rows if r['image_id'] not in done]
    model.eval()
    with open(out_path, 'a', encoding='utf-8') as f:
        for i in tqdm(range(0, len(todo), batch_size), desc=out_path.name):
            batch = todo[i:i + batch_size]
            convs = [build_messages(Image.open(DATA / r['image_path']).convert('RGB'), prompt) for r in batch]
            inputs = processor.apply_chat_template(convs, tokenize=True, add_generation_prompt=True,
                                                   return_dict=True, return_tensors='pt',
                                                   padding=True).to(model.device)
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
            raws = processor.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
            for r, raw in zip(batch, raws):
                f.write(json.dumps({'image_id': r['image_id'], 'raw': raw}, ensure_ascii=False) + '\n')
            f.flush()
    preds = [json.loads(l) for l in open(out_path, encoding='utf-8')]
    ids = [p['image_id'] for p in preds]
    assert len(ids) == len(set(ids)), 'duplicate image_id in predictions'
    assert {r['image_id'] for r in rows} <= set(ids), 'missing predictions'
    return preds

t0 = time.time()
model, processor = load_model()
print(f'model loaded in {time.time() - t0:.0f}s | GPU allocated {torch.cuda.memory_allocated() / 1e9:.2f} GB')
print('visual dtype:', next(model.model.visual.parameters()).dtype, '| lm_head dtype:', model.lm_head.weight.dtype)

In [ ]:
# E0 inference: quick sanity on 4 val images, then the full test split (311 images) -> work/e0.jsonl
t0 = time.time()
sanity = predict(model, processor, splits['val'][:4], 'work/e0_sanity_val.jsonl')
print(f'4 val images in {time.time() - t0:.1f}s')
for p in sanity:
    print(p['image_id'], '->', repr(p['raw'][:300]))

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
e0_preds = predict(model, processor, splits['test'], 'work/e0.jsonl')
print(f"E0: {len(e0_preds)} predictions in {(time.time() - t0) / 60:.1f} min | "
      f"peak VRAM {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

In [ ]:
# Score E0 with the referee, then inspect raw outputs (for REPORT question 2)
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json
# same predictions read as PIXEL coordinates: does the base model answer in pixels instead of 0-1000? (question 3)
!python scripts/score.py --pred work/e0.jsonl --name E0-as-pixel --coord pixel --out work/e0_as_pixel.json

e0 = [json.loads(l) for l in open('work/e0.jsonl', encoding='utf-8')]
parsed = [parse_output(p['raw']) for p in e0]
boxes = [b for ps in parsed if ps for b in ps]
print(f"\nparse_valid_rate={sum(ps is not None for ps in parsed) / len(e0):.3f} | "
      f"fenced with ```: {sum('```' in p['raw'] for p in e0)} | "
      f"empty []: {sum(ps == [] for ps in parsed)} | predicted boxes: {len(boxes)} (GT test boxes: 583)")
print('predicted labels:', dict(Counter(b['label'] for b in boxes)))
if boxes:
    mx = [max(b['bbox']) for b in boxes]
    print(f"max coordinate per box: median={sorted(mx)[len(mx) // 2]:.0f}, max={max(mx):.0f}; "
          f"share of boxes entirely within 640x360: {sum(b['bbox'][2] <= 640 and b['bbox'][3] <= 360 for b in boxes) / len(boxes):.2f}")
print('\n--- 5 raw outputs ---')
for p in e0[:5]:
    print(p['image_id'], '->', repr(p['raw'][:400]))
bad = [p for p, ps in zip(e0, parsed) if ps is None]
print(f'\n--- {len(bad)} unparseable outputs, first 3 ---')
for p in bad[:3]:
    print(p['image_id'], '->', repr(p['raw'][:400]))

In [ ]:
# E0 resource log (peak VRAM was reset just before the test run and nothing used the GPU since)
import os
e0_peak = torch.cuda.max_memory_allocated() / 1e9
# sanity file was last written right before the test run started; e0.jsonl last written when it ended
e0_min = (os.path.getmtime('work/e0.jsonl') - os.path.getmtime('work/e0_sanity_val.jsonl')) / 60
json.dump({'peak_vram_gb': round(e0_peak, 2), 'infer_minutes': round(e0_min, 1)}, open('work/e0_run.json', 'w'))
print(f"E0 inference: peak VRAM {e0_peak:.2f} GB | {e0_min:.1f} min for 311 test images")
!ls -la report/evidence work

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# YOUR CODE HERE: encode_example() và collate() (GUIDE bước 4)

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# YOUR CODE HERE: apply_setup(model, setup) (GUIDE bước 5)

## 6. Vòng huấn luyện

In [ ]:
# YOUR CODE HERE: vòng train, lưu adapter + run.json (GUIDE bước 6)

## 7. Suy luận và chấm E1–E3

In [ ]:
# YOUR CODE HERE: suy luận test → work/eN.jsonl rồi chạy scripts/score.py với --meta (GUIDE bước 7)

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# YOUR CODE HERE: GUIDE bước 8 và 9